70/15/15 train/validation/test split
Example random split sizes: 424 91 91 (Seed 42)

Dataset shape: (606, 28)
Number of situations: 6

ModelConfig(experiment_name='bert_base_uncased', model_name='bert-base-uncased', batch_size=8, learning_rate=1e-05, weight_decay=0.01, dropout=0.4, max_length=256, num_epochs=12, patience=3, local_files_only=True)

 ModelConfig(experiment_name='bert_large_uncased', model_name='google-bert/bert-large-uncased', batch_size=4, learning_rate=5e-06, weight_decay=0.01, dropout=0.5, max_length=256, num_epochs=12, patience=3, local_files_only=False)


 ModelConfig(experiment_name='modernbert_base', model_name='answerdotai/ModernBERT-base', batch_size=8, learning_rate=1e-05, weight_decay=0.01, dropout=0.4, max_length=256, num_epochs=12, patience=3, local_files_only=False)



In [ ]:
from __future__ import annotations

import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable
import pandas as pd

In [ ]:
PROJECT_DIR = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'data' / 'annotate-lime.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')
DATA_DIR = PROJECT_DIR / 'data'

OUTPUT_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'baseline'
SUMMARY_DIR = OUTPUT_DIR / 'summaries'
TUNING_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'lr_epoch_tuning'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 8 strategies
REAPPRAISAL_TECHNIQUES = ['[GM]', '[PR]', '[A]', '[EJ]', '[N]', '[UC]', '[V]', '[H]']
LABEL_COLS = [tech.strip('[]') for tech in REAPPRAISAL_TECHNIQUES]

LABEL_NAMES = {
    'GM': 'growth mindset',
    'PR': 'positive repurposing',
    'A': 'aberration',
    'EJ': 'external justification',
    'N': 'normalizing',
    'UC': 'unwarranted conclusions',
    'V': 'validation',
    'H': 'humor',
}

print('Project directory:', PROJECT_DIR)
print('Output directory:', OUTPUT_DIR)
print('Labels:', LABEL_COLS)

summary_path = SUMMARY_DIR / 'random_splits_binomial_nll_by_model.csv'
if summary_path.exists():
    display(
        pd.read_csv(summary_path)
        .sort_values(['macro_mae_mean', 'mean_pearson_mean'], ascending=[True, False])
    )
else:
    print('No aggregate summary yet:', summary_path)

In [ ]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support


MPL_CACHE_DIR = PROJECT_DIR / '.matplotlib_cache'
MPL_CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE_DIR))


RESULTS_DIR = OUTPUT_DIR

MODEL_DISPLAY_NAMES = {
    'bert_base_uncased': 'BERT-base',
    'bert_large_uncased': 'BERT-large',
    'modernbert_base': 'ModernBERT-base',
    'roberta_base': 'RoBERTa-base',
    'roberta_large': 'RoBERTa-large',
}

MODEL_COLORS = {
    'BERT-base': '#3b82f6',
    'BERT-large': '#0f766e',
    'ModernBERT-base': '#f59e0b',
    'RoBERTa-base': '#8b5cf6',
    'RoBERTa-large': '#dc2626',
}

plt.rcParams.update({
    'figure.figsize': (8, 4.5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

summary_df = pd.read_csv(SUMMARY_DIR / 'random_splits_binomial_nll.csv')
summary_df['Model'] = summary_df['experiment_name'].map(MODEL_DISPLAY_NAMES)

summary_by_model_df = pd.read_csv(SUMMARY_DIR / 'random_splits_binomial_nll_by_model.csv')
summary_by_model_df['Model'] = summary_by_model_df['experiment_name'].map(MODEL_DISPLAY_NAMES)

summary_df[['Model', 'run_id', 'loss_mode']].sort_values('Model')


In [ ]:
# Table 1: Main model comparison.
# Lower validation loss, MAE, and RMSE are better.
# Higher correlations and F1 are better.
main_results_table = (
    summary_df[[
        'Model',
        'best_epoch',
        'best_val_loss',
        'macro_mae',
        'macro_rmse',
        'mean_pearson',
        'mean_spearman',
        'majority_tuned_macro_f1',
        'strong_presence_tuned_macro_f1',
    ]]
    .rename(columns={
        'best_epoch': 'Best epoch',
        'best_val_loss': 'Validation loss',
        'macro_mae': 'Macro MAE',
        'macro_rmse': 'Macro RMSE',
        'mean_pearson': 'Mean Pearson',
        'mean_spearman': 'Mean Spearman',
        'majority_tuned_macro_f1': 'Majority Macro F1',
        'strong_presence_tuned_macro_f1': 'Strong Presence Macro F1',
    })
    .sort_values('Macro MAE')
    .reset_index(drop=True)
)

display(main_results_table.round(3))


### Key Results Table

This table collects the most important information across the original runner and LR/epoch-tuning runs: model, seed, learning rate, scheduler, best epoch, validation loss, soft-label metrics, and thresholded F1.


In [ ]:
# Key results table across original and tuned runs.
import statistics

KEY_RESULTS_DIR = TUNING_DIR
KEY_RESULTS_DIR.mkdir(exist_ok=True)


def key_base_experiment_name(experiment_name: str) -> str:
    return experiment_name.split('_lr_')[0]


def load_key_result(metrics_path: Path, source: str) -> dict:
    payload = json.loads(metrics_path.read_text())
    config = payload.get('config', {})
    experiment_name = payload['experiment_name']
    base_name = key_base_experiment_name(experiment_name)
    soft_summary = payload['soft_summary']
    binary_metrics = payload['binary_scheme_metrics']

    return {
        'Source': source,
        'Model': MODEL_DISPLAY_NAMES.get(base_name, base_name),
        'Experiment': experiment_name,
        'Seed': payload['run_id'].replace('random_seed', ''),
        'Learning rate': config.get('learning_rate', 'original'),
        'Scheduler': config.get('scheduler_type', 'linear'),
        'Best epoch': payload['best_epoch'],
        'Validation loss': payload['best_val_loss'],
        'Macro MAE': soft_summary['macro_mae'],
        'Macro RMSE': soft_summary['macro_rmse'],
        'Mean Pearson': soft_summary['mean_pearson'],
        'Mean Spearman': soft_summary['mean_spearman'],
        'Majority Macro F1': binary_metrics['majority']['tuned_binary_metrics']['macro_f1'],
        'Strong Presence Macro F1': binary_metrics['strong_presence']['tuned_binary_metrics']['macro_f1'],
        'Weak Presence Macro F1': binary_metrics['weak_presence']['tuned_binary_metrics']['macro_f1'],
    }


key_result_rows = []
for metrics_path in sorted(RESULTS_DIR.glob('*/*/*_random_seed*_metrics.json')):
    key_result_rows.append(load_key_result(metrics_path, 'Original runner'))
for metrics_path in sorted(KEY_RESULTS_DIR.glob('*_metrics.json')):
    key_result_rows.append(load_key_result(metrics_path, 'LR/epoch tuning'))

key_results_table = (
    pd.DataFrame(key_result_rows)
    .sort_values(['Macro MAE', 'Validation loss'], ascending=[True, True])
    .reset_index(drop=True)
)
key_results_table.insert(0, 'Rank', range(1, len(key_results_table) + 1))
key_results_table.to_csv(KEY_RESULTS_DIR / 'key_results_table.csv', index=False)

important_cols = [
    'Rank',
    'Source',
    'Model',
    'Seed',
    'Learning rate',
    'Scheduler',
    'Best epoch',
    'Validation loss',
    'Macro MAE',
    'Mean Pearson',
    'Majority Macro F1',
    'Strong Presence Macro F1',
]

display(key_results_table[important_cols].head(15).round(4))

# Mean table by configuration. Prefer this table when a configuration has multiple seeds.
config_group_cols = ['Source', 'Model', 'Experiment', 'Learning rate', 'Scheduler']
key_results_by_config = (
    key_results_table
    .groupby(config_group_cols, as_index=False)
    .agg(
        n_seeds=('Seed', 'count'),
        seeds=('Seed', lambda values: ','.join(sorted(map(str, values)))),
        best_epoch_mean=('Best epoch', 'mean'),
        val_loss_mean=('Validation loss', 'mean'),
        val_loss_sd=('Validation loss', 'std'),
        macro_mae_mean=('Macro MAE', 'mean'),
        macro_mae_sd=('Macro MAE', 'std'),
        mean_pearson_mean=('Mean Pearson', 'mean'),
        mean_pearson_sd=('Mean Pearson', 'std'),
        majority_macro_f1_mean=('Majority Macro F1', 'mean'),
        majority_macro_f1_sd=('Majority Macro F1', 'std'),
        strong_presence_macro_f1_mean=('Strong Presence Macro F1', 'mean'),
        strong_presence_macro_f1_sd=('Strong Presence Macro F1', 'std'),
    )
    .sort_values(['macro_mae_mean', 'val_loss_mean'], ascending=[True, True])
    .reset_index(drop=True)
)
key_results_by_config.insert(0, 'Rank', range(1, len(key_results_by_config) + 1))
key_results_by_config.to_csv(KEY_RESULTS_DIR / 'key_results_by_config.csv', index=False)

display(
    key_results_by_config[[
        'Rank',
        'Source',
        'Model',
        'Learning rate',
        'Scheduler',
        'n_seeds',
        'seeds',
        'best_epoch_mean',
        'val_loss_mean',
        'macro_mae_mean',
        'mean_pearson_mean',
        'majority_macro_f1_mean',
        'strong_presence_macro_f1_mean',
    ]]
    .head(12)
    .round(4)
)


In [ ]:
# Figure 1: Macro MAE by model.
# Lower is better.
plot_df = main_results_table.sort_values('Macro MAE')

fig, ax = plt.subplots(figsize=(7, 4))
colors = [MODEL_COLORS[model] for model in plot_df['Model']]
bars = ax.bar(plot_df['Model'], plot_df['Macro MAE'], color=colors)

ax.set_title('Macro MAE by Model')
ax.set_ylabel('Macro MAE')
ax.set_xlabel('Model')
ax.set_ylim(0, plot_df['Macro MAE'].max() * 1.25)

for bar in bars:
    value = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()



In [ ]:
# Prepare per-strategy soft-label metrics.
soft_metric_tables = []

for metrics_path in sorted(RESULTS_DIR.glob('*/*/*_random_seed42_soft_metrics.csv')):
    experiment_name = metrics_path.name.replace('_random_seed42_soft_metrics.csv', '')
    metrics = pd.read_csv(metrics_path)
    metrics['Model'] = MODEL_DISPLAY_NAMES[experiment_name]
    soft_metric_tables.append(metrics)

soft_metrics_all = pd.concat(soft_metric_tables, ignore_index=True)

mae_by_strategy = (
    soft_metrics_all
    .pivot(index='label', columns='Model', values='mae')
    .loc[LABEL_COLS, ['BERT-base', 'BERT-large', 'ModernBERT-base']]
)
mae_by_strategy.index = [f'{label} - {LABEL_NAMES[label]}' for label in mae_by_strategy.index]

display(mae_by_strategy.round(3))


In [ ]:
# Figure 2: Per-strategy MAE heatmap.
# Lower values are better.
fig, ax = plt.subplots(figsize=(8, 15))
image = ax.imshow(mae_by_strategy.values, cmap='Blues_r')

ax.set_title('Per-Strategy MAE by Model')
ax.set_xticks(range(len(mae_by_strategy.columns)))
ax.set_xticklabels(mae_by_strategy.columns)
ax.set_yticks(range(len(mae_by_strategy.index)))
ax.set_yticklabels(mae_by_strategy.index)

for i in range(mae_by_strategy.shape[0]):
    for j in range(mae_by_strategy.shape[1]):
        ax.text(j, i, f'{mae_by_strategy.iloc[i, j]:.3f}', ha='center', va='center', color='black')

cbar = fig.colorbar(image, ax=ax)
cbar.set_label('MAE')
ax.grid(False)
plt.tight_layout()
plt.show()


In [ ]:
# Table 2: Secondary binary F1 results.
# These are not the primary training target; they turn soft scores into binary predictions.
binary_f1_table = (
    summary_df[[
        'Model',
        'weak_presence_tuned_macro_f1',
        'majority_tuned_macro_f1',
        'strong_presence_tuned_macro_f1',
    ]]
    .rename(columns={
        'weak_presence_tuned_macro_f1': 'Weak Presence Macro F1',
        'majority_tuned_macro_f1': 'Majority Macro F1',
        'strong_presence_tuned_macro_f1': 'Strong Presence Macro F1',
    })
    .sort_values('Majority Macro F1', ascending=False)
    .reset_index(drop=True)
)

display(binary_f1_table.round(3))


In [ ]:
# Figure 3: Tuned binary macro F1 by binary target definition.
# Higher is better.
plot_df = binary_f1_table.set_index('Model')
metric_cols = ['Weak Presence Macro F1', 'Majority Macro F1', 'Strong Presence Macro F1']

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df[metric_cols].plot(kind='bar', ax=ax, color=['#93c5fd', '#0f766e', '#f59e0b'])

ax.set_title('Binary Macro F1')
ax.set_ylabel('Macro F1')
ax.set_xlabel('Model')
ax.set_ylim(0, 1)
ax.legend(title='Binary target')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# Table 3: Training summary across original and LR/epoch-tuning runs.
TUNING_RESULTS_DIR = TUNING_DIR

BASE_MODEL_DISPLAY_NAMES = {
    'bert_base_uncased': 'BERT-base',
    'bert_large_uncased': 'BERT-large',
    'modernbert_base': 'ModernBERT-base',
    'roberta_base': 'RoBERTa-base',
    'roberta_large': 'RoBERTa-large',
}


def parse_history_filename(history_path: Path, source: str) -> dict:
    name = history_path.name.replace('_training_history.csv', '')
    experiment_name, seed_text = name.rsplit('_random_seed', 1)
    run_id = f'random_seed{seed_text}'
    base_experiment_name = experiment_name.split('_lr_')[0]
    lr_label = 'original'
    scheduler_type = 'linear'
    if '_lr_' in experiment_name:
        lr_part = experiment_name.split('_lr_', 1)[1]
        if '_sched_' in lr_part:
            lr_label, scheduler_slug = lr_part.split('_sched_', 1)
            scheduler_type = scheduler_slug.replace('-', '_')
        else:
            lr_label = lr_part
        lr_label = lr_label.replace('_', '-')
    return {
        'source': source,
        'experiment_name': experiment_name,
        'base_experiment_name': base_experiment_name,
        'Model': BASE_MODEL_DISPLAY_NAMES.get(base_experiment_name, base_experiment_name),
        'run_id': run_id,
        'learning_rate': lr_label,
        'scheduler_type': scheduler_type,
    }


def load_training_histories() -> pd.DataFrame:
    history_frames = []
    history_sources = [
        ('Original runner', RESULTS_DIR.glob('*/*/*_random_seed*_training_history.csv')),
        ('LR/epoch tuning', TUNING_RESULTS_DIR.glob('*_random_seed*_training_history.csv')),
    ]

    for source, paths in history_sources:
        for history_path in sorted(paths):
            meta = parse_history_filename(history_path, source)
            history = pd.read_csv(history_path)
            for key, value in meta.items():
                history[key] = value
            history['run_label'] = history.apply(
                lambda row: f"{row['Model']} | {row['learning_rate']} | {row['scheduler_type']} | {row['run_id']} | {row['source']}",
                axis=1,
            )
            history_frames.append(history)

    if not history_frames:
        return pd.DataFrame()
    return pd.concat(history_frames, ignore_index=True)


training_history_all = load_training_histories()

training_rows = []
for run_label, history in training_history_all.groupby('run_label'):
    best_row = history.loc[history['val_loss'].idxmin()]
    final_row = history.sort_values('epoch').iloc[-1]
    training_rows.append({
        'Model': best_row['Model'],
        'Source': best_row['source'],
        'Run': best_row['run_id'],
        'Learning rate': best_row['learning_rate'],
        'Scheduler': best_row['scheduler_type'],
        'Best epoch': int(best_row['epoch']),
        'Best validation loss': best_row['val_loss'],
        'Final training loss': final_row['train_loss'],
        'Final validation loss': final_row['val_loss'],
        'Epochs run': int(final_row['epoch']),
    })

training_summary_table = (
    pd.DataFrame(training_rows)
    .sort_values(['Best validation loss', 'Model', 'Learning rate', 'Scheduler'])
    .reset_index(drop=True)
)

display(training_summary_table.round(3))


In [ ]:
# Figure 4: Training and validation loss curves across original and LR/epoch-tuning runs.
if training_history_all.empty:
    print('No training histories found.')
else:
    # Keep the combined overview readable by plotting validation loss only.
    validation_plot_df = training_history_all.copy()
    validation_plot_df['line_label'] = validation_plot_df.apply(
        lambda row: f"{row['Model']} {row['learning_rate']} {row['scheduler_type']} {row['run_id']}",
        axis=1,
    )

    fig, ax = plt.subplots(figsize=(11, 6))
    for line_label, history in validation_plot_df.groupby('line_label'):
        history = history.sort_values('epoch')
        base_model = history['Model'].iloc[0]
        source = history['source'].iloc[0]
        linestyle = '-' if source == 'LR/epoch tuning' else '--'
        color = MODEL_COLORS.get(base_model, '#6b7280')
        ax.plot(
            history['epoch'],
            history['val_loss'],
            linestyle=linestyle,
            linewidth=2,
            alpha=0.85,
            color=color,
            label=line_label,
        )
        best_row = history.loc[history['val_loss'].idxmin()]
        ax.scatter(best_row['epoch'], best_row['val_loss'], color=color, s=45, zorder=3)

    ax.set_title('Validation Loss Across Original and Tuned BERT Runs')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation loss')
    ax.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()

    # A focused view of training vs validation for the strongest tuned run so far.
    best_run = training_summary_table.iloc[0]
    focused_history = training_history_all[
        (training_history_all['Model'] == best_run['Model'])
        & (training_history_all['source'] == best_run['Source'])
        & (training_history_all['run_id'] == best_run['Run'])
        & (training_history_all['learning_rate'] == best_run['Learning rate'])
        & (training_history_all['scheduler_type'] == best_run['Scheduler'])
    ].sort_values('epoch')

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(focused_history['epoch'], focused_history['train_loss'], marker='o', label='Training loss')
    ax.plot(focused_history['epoch'], focused_history['val_loss'], marker='o', label='Validation loss')
    ax.axvline(best_run['Best epoch'], color='black', linestyle='--', alpha=0.55, label=f"best epoch {int(best_run['Best epoch'])}")
    ax.set_title(f"Best Run Learning Curve: {best_run['Model']} {best_run['Learning rate']} {best_run['Scheduler']} {best_run['Run']}")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Table 4: Per-label majority F1 and test support.
# Low support means a label has few positive test examples, so F1 is unstable.
majority_f1_rows = []

for predictions_path in sorted(RESULTS_DIR.glob('*/*/*_random_seed42_test_predictions.csv')):
    experiment_name = predictions_path.name.replace('_random_seed42_test_predictions.csv', '')
    predictions = pd.read_csv(predictions_path)

    y_true = predictions[[f'{label}_majority_true_binary' for label in LABEL_COLS]].to_numpy()
    y_pred = predictions[[f'{label}_majority_pred_binary' for label in LABEL_COLS]].to_numpy()

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    for label, label_f1, label_support in zip(LABEL_COLS, f1, support):
        majority_f1_rows.append({
            'Label': label,
            'Strategy': LABEL_NAMES[label],
            'Test positives': int(label_support),
            'Model': MODEL_DISPLAY_NAMES[experiment_name],
            'F1': label_f1,
        })

majority_f1_long = pd.DataFrame(majority_f1_rows)
majority_f1_table = majority_f1_long.pivot_table(
    index=['Label', 'Strategy', 'Test positives'],
    columns='Model',
    values='F1',
).reset_index()
majority_f1_table = majority_f1_table.set_index('Label').loc[LABEL_COLS].reset_index()
majority_f1_table = majority_f1_table[
    ['Label', 'Strategy', 'Test positives', 'BERT-base', 'BERT-large', 'ModernBERT-base']
]

display(majority_f1_table.round(3))


In [ ]:
# Figure 5: Per-label majority F1 heatmap.
# Higher values are better.
f1_heatmap = majority_f1_table.set_index('Label')[['BERT-base', 'BERT-large', 'ModernBERT-base']]

fig, ax = plt.subplots(figsize=(7, 10))
image = ax.imshow(f1_heatmap.values, cmap='Blues', vmin=0, vmax=1)

ax.set_title('Per-Label Majority F1 by Model')
ax.set_xticks(range(len(f1_heatmap.columns)))
ax.set_xticklabels(f1_heatmap.columns)
ax.set_yticks(range(len(f1_heatmap.index)))
ax.set_yticklabels(f1_heatmap.index)

for i in range(f1_heatmap.shape[0]):
    for j in range(f1_heatmap.shape[1]):
        ax.text(j, i, f'{f1_heatmap.iloc[i, j]:.3f}', ha='center', va='center', color='black')

cbar = fig.colorbar(image, ax=ax)
cbar.set_label('F1')
ax.grid(False)
plt.tight_layout()
plt.show()
